# 🧹 Data Cleaning & Preprocessing Pipeline
**Objective:** Address data quality issues identified during the EDA phase. This includes clamping bounded behavioral variables, rounding synthetic oversampling artifacts (SMOTE float numbers), standardizing biometric metrics, and exporting a clean dataset for downstream clustering modeling.

In [1]:
import pandas as pd
import numpy as np

# Load the raw and synthetic dataset
df = pd.read_csv('../data/raw/ObesityDataSet_raw_and_data_sinthetic.csv')

# Display initial dimensions
print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")

Dataset loaded successfully with 2111 rows and 17 columns.


## 1. Clamping & Cleaning Behavioral Variables

Based on official dataset technical documentation, survey responses regarding eating habits, physical activity, and technology usage follow strict bounded domain ranges. 

Due to SMOTE linear interpolation, these discrete features contain high-precision continuous floating points as well as out-of-range values (e.g., `NCP` values exceeding 3.0).

We apply a three-step cleaning transformation:
1. **Rounding:** Round values to the nearest integer to restore discrete frequency categories.
2. **Clamping (`clip`):** Enforce strict upper and lower domain bounds according to survey definitions.
3. **Casting (`astype(int)`):** Convert cleaned metrics into clean integer data types.

In [2]:
# Define valid technical domain ranges: 'Column': (min_value, max_value)
technical_ranges = {
    'FCVC': (1, 3),  # Frequency of consumption of vegetables
    'NCP':  (1, 3),  # Number of main meals
    'CH2O': (1, 3),  # Daily consumption of water
    'FAF':  (0, 3),  # Physical activity frequency
    'TUE':  (0, 2)   # Time using technology devices
}

# Explicitly fix NCP upper bound before transformation
df.loc[df['NCP'] > 3, 'NCP'] = 3

# Apply transformation loop: Round -> Clip -> Cast to int
for col, (min_val, max_val) in technical_ranges.items():
    df[col] = df[col].round(0).clip(lower=min_val, upper=max_val).astype(int)

# Verify transformed ranges and data types
print("--- Behavioral Features Post-Cleaning Verification ---")
for col in technical_ranges.keys():
    print(f"{col:<6} | Min: {df[col].min()} | Max: {df[col].max()} | Data Type: {df[col].dtype}")

--- Behavioral Features Post-Cleaning Verification ---
FCVC   | Min: 1 | Max: 3 | Data Type: int64
NCP    | Min: 1 | Max: 3 | Data Type: int64
CH2O   | Min: 1 | Max: 3 | Data Type: int64
FAF    | Min: 0 | Max: 3 | Data Type: int64
TUE    | Min: 0 | Max: 2 | Data Type: int64


## 2. Biometric Data Standardization (`Age`, `Height`, `Weight`)

Biometric variables were also affected by synthetic oversampling, resulting in unrealistically detailed floating-point numbers. Standard clinical precision rules are applied:
* **`Age`:** Rounded and converted to integer (discrete years).
* **`Height`:** Rounded to 2 decimal places (standard clinical measurement in meters, e.g., `1.75`).
* **`Weight`:** Rounded to 1 decimal place (standard weight scale measurement in kilograms, e.g., `85.4`).

In [3]:
# Standardize biometric metrics precision
df['Age'] = df['Age'].round(0).astype(int)
df['Height'] = df['Height'].round(2)
df['Weight'] = df['Weight'].round(1)

# Verify biometric features sample
print("--- Biometric Metrics Verification ---")
display(df[['Age', 'Height', 'Weight']].head())

--- Biometric Metrics Verification ---


,Age,Height,Weight
0,21,1.62,64.0
1,21,1.52,56.0
2,23,1.80,77.0
3,27,1.80,87.0
4,22,1.78,89.8


## 3. Dataset Export

The cleaned dataset is saved into the project's processed data folder (`data/processed/`) to maintain a clean separation between raw inputs and cleaned outputs.

In [4]:
# Export cleaned dataset to processed directory
output_path = '../data/interim/ObesityDataSet_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Cleaned dataset successfully exported to: {output_path}")

Cleaned dataset successfully exported to: ../data/interim/ObesityDataSet_cleaned.csv
